# Milestone 2

In [1]:
from transformers import AutoTokenizer, pipeline
import pandas as pd
import numpy as np

In [2]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="../dataset/train.csv",split = "train")

In [3]:
def add_combined_text(example):
    example["combined_text"] = str(example["prompt"]) + " " + str(example["A"])
    return example
 
dataset = dataset.map(add_combined_text)
 
print(f"New columns: {dataset.column_names}")

combined_text_51 = dataset[51]["combined_text"]
 
print(f"Row index 51 — combined_text:\n{combined_text_51}")
print(f"Character length at index 51: {len(combined_text_51)}")


New columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer', 'combined_text']
Row index 51 — combined_text:
Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
Character length at index 51: 614


In [4]:
train = pd.read_csv('../dataset/train.csv')

In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(f"bert-base-uncased vocab size: {tokenizer.vocab_size}")

bert-base-uncased vocab size: 30522


In [6]:
sep_token_id = tokenizer.sep_token_id
print(f"[SEP] token ID: {sep_token_id}")

[SEP] token ID: 102


In [7]:
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModel

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-uncased")

Some weights of the model checkpoint at google-bert/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [8]:
encodings = tokenizer(
    train["prompt"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

In [9]:
print(f"input_ids tensor shape: {encodings['input_ids'].shape}")

input_ids tensor shape: torch.Size([2000, 128])


In [10]:
hidden_size = model.config.hidden_size
num_heads = model.config.num_attention_heads

head_dim = hidden_size // num_heads

print("Each attention head dimension:", head_dim)

Each attention head dimension: 64


In [11]:
import torch

model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

text = train.loc[0, "prompt"]
inputs = tokenizer(text, return_tensors="pt")

print(f"\nRow 0 prompt  : {text}")
print(f"Token count   : {inputs['input_ids'].shape[1]}")

with torch.no_grad():
    outputs = model(**inputs)

print(f"last_hidden_state shape: {outputs.last_hidden_state.shape}")


Row 0 prompt  : Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
Token count   : 31
last_hidden_state shape: torch.Size([1, 31, 768])


In [12]:
cls_vector = outputs.last_hidden_state[0, 0, :]
 
first_5_values = cls_vector[:5].tolist()
sum_first_5    = sum(first_5_values)
 
print(f"First 5 values of [CLS] vector: {[round(v, 6) for v in first_5_values]}")
print(f"Sum of first 5 values  : {round(sum_first_5, 4)}")

First 5 values of [CLS] vector: [-0.467665, -0.075445, -0.201901, -0.007064, -0.448023]
Sum of first 5 values  : -1.2001


In [13]:
model_attn = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
model_attn.eval()
 
exact_string = "Light-ion fusion is a technique."
inputs_attn  = tokenizer(exact_string, return_tensors="pt")

tokens = tokenizer.convert_ids_to_tokens(inputs_attn["input_ids"][0])
print(f"Tokens: {tokens}")
 
fusion_index = tokens.index("fusion")
print(f"'fusion' is at token index: {fusion_index}")
 
with torch.no_grad():
    outputs_attn = model_attn(**inputs_attn)

last_layer_attn = outputs_attn.attentions[-1]
head_0_attn     = last_layer_attn[0, 0, :, :]

cls_to_fusion   = head_0_attn[0, fusion_index].item()
 
print(f"[CLS] → 'fusion' attention weight: {round(cls_to_fusion, 4)}")

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
'fusion' is at token index: 4
[CLS] → 'fusion' attention weight: 0.1025


In [14]:
from sentence_transformers import SentenceTransformer, util

In [15]:
minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
 
prompt_0  = dataset[0]["prompt"]
option_b0 = dataset[0]["B"]
 
emb_prompt   = minilm.encode(prompt_0,  convert_to_tensor=True)
emb_option_b = minilm.encode(option_b0, convert_to_tensor=True)
 
cos_sim_q6 = util.cos_sim(emb_prompt, emb_option_b).item()
 
print(f"Row 0 prompt  : {prompt_0}")
print(f"Row 0 option B: {option_b0}")
print(f"Cosine similarity (prompt vs Option B): {round(cos_sim_q6, 4)}")

Row 0 prompt  : Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
Row 0 option B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Cosine similarity (prompt vs Option B): 0.7658


In [16]:
def map_at_3(ground_truth, predictions):
    for i, p in enumerate(predictions[:3]):
        if p == ground_truth:
            return 1.0 / (i + 1)
    return 0.0

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity 

In [18]:
all_text_corpus = []
CHOICES = ["A", "B", "C", "D", "E"]

for row in dataset:
    all_text_corpus.append(str(row["prompt"]))
    for c in CHOICES:
        all_text_corpus.append(str(row[c]))
 
tfidf_vec = TfidfVectorizer(stop_words="english")
tfidf_vec.fit(all_text_corpus)
 
prompts_list = [str(row["prompt"]) for row in dataset]
p_vecs       = tfidf_vec.transform(prompts_list)
opt_vecs     = {c: tfidf_vec.transform([str(row[c]) for row in dataset]) for c in CHOICES}
 
tfidf_top3_all = []
tfidf_map3_scores = []
 
for i, row in enumerate(dataset):
    sims = [(c, cosine_similarity(p_vecs[i], opt_vecs[c][i])[0][0]) for c in CHOICES]
    sims.sort(key=lambda x: x[1], reverse=True)
    top3 = [x[0] for x in sims[:3]]
    tfidf_top3_all.append(top3)
    tfidf_map3_scores.append(map_at_3(row["answer"], top3))
 
print(f"TF-IDF MAP@3 (for reference): {round(np.mean(tfidf_map3_scores), 4)}")

TF-IDF MAP@3 (for reference): 0.3119


In [19]:
all_prompt_texts = [str(row["prompt"]) for row in dataset]
all_option_texts = {c: [str(row[c]) for row in dataset] for c in CHOICES}
 
prompt_embs = minilm.encode(all_prompt_texts, batch_size=64,
                             show_progress_bar=True, convert_to_tensor=True)
option_embs = {c: minilm.encode(all_option_texts[c], batch_size=64,
                                 show_progress_bar=False, convert_to_tensor=True)
               for c in CHOICES}
 
minilm_top3_all  = []
minilm_map3_scores = []
tfidf_miss_minilm_hit = 0
 
for i, row in enumerate(dataset):
    sims = [(c, util.cos_sim(prompt_embs[i], option_embs[c][i]).item()) for c in CHOICES]
    sims.sort(key=lambda x: x[1], reverse=True)
    top3 = [x[0] for x in sims[:3]]
    minilm_top3_all.append(top3)
    minilm_map3_scores.append(map_at_3(row["answer"], top3))
 
    answer = row["answer"]
    tfidf_miss = answer not in tfidf_top3_all[i]
    minilm_hit = answer in top3
    if tfidf_miss and minilm_hit:
        tfidf_miss_minilm_hit += 1
 
print(f"MiniLM MAP@3: {round(np.mean(minilm_map3_scores), 4)}")
print(f"TF-IDF miss but MiniLM hit (count)    : {tfidf_miss_minilm_hit}")

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

MiniLM MAP@3: 0.4231
TF-IDF miss but MiniLM hit (count)    : 462


In [20]:
row_1        = dataset[1]
prompt_row1  = str(row_1["prompt"])
candidates   = [str(row_1["A"]), str(row_1["B"]), str(row_1["C"])]
 
print(f"Row 1 prompt    : {prompt_row1}")
print(f"Candidate labels: {candidates}")
 
zsc = pipeline("zero-shot-classification", device=-1) 
 
result_q8 = zsc(prompt_row1, candidate_labels=candidates, multi_label=False)
 
print(f"Full result: {result_q8}")
top_score_q8 = result_q8["scores"][0]
sum_scores_q8 = sum(result_q8["scores"])
print(f"Top-ranked option probability: {round(top_score_q8, 4)}")
print(f"Sum of all 3 probabilities (Softmax) : {round(sum_scores_q8, 4)}")

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.


Row 1 prompt    : What is accelerator-based light-ion fusion?
Candidate labels: ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce 

Device set to use cpu


Full result: {'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to ind

In [21]:
result_q9    = zsc(prompt_row1, candidate_labels=candidates, multi_label=True)
 
sum_scores_q9 = sum(result_q9["scores"])
abs_diff      = abs(sum_scores_q8 - sum_scores_q9)
 
print(f"Full result (multi_label=True): {result_q9}")
print(f"Sum of 3 probabilities (Sigmoid)   : {round(sum_scores_q9, 4)}")
print(f"Sum of 3 probabilities (Softmax)   : {round(sum_scores_q8, 4)}")
print(f"Absolute difference          : {round(abs_diff, 4)}")

Full result (multi_label=True): {'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies

In [22]:
result_q9    = zsc(prompt_row1, candidate_labels=candidates, multi_label=True)
 
sum_scores_q9 = sum(result_q9["scores"])
abs_diff      = abs(sum_scores_q8 - sum_scores_q9)
 
print(f"Full result (multi_label=True): {result_q9}")
print(f"Sum of 3 probabilities (Sigmoid)   : {round(sum_scores_q9, 4)}")
print(f"Sum of 3 probabilities (Softmax)   : {round(sum_scores_q8, 4)}")
print(f"Absolute difference: {round(abs_diff, 4)}")

Full result (multi_label=True): {'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies

In [25]:
row_0   = dataset[0]
prompt  = str(row_0["prompt"])
opt_a   = str(row_0["A"])
opt_b   = str(row_0["B"])
 
flan_input = (
    f"Question: {prompt}. "
    f"Is the correct answer A: {opt_a} or B: {opt_b}? "
    f"Answer with just the letter A or B."
)
 
print(f"Flan-T5 input string:\n{flan_input}")
 
flan = pipeline("text2text-generation", model="google/flan-t5-small", device=-1)
flan_output = flan(flan_input, max_new_tokens=5)
 
generated_text = flan_output[0]["generated_text"]
print(f"Flan-T5 output: '{generated_text}'")

Flan-T5 input string:
Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.


Device set to use cpu


Flan-T5 output: 'B'
